In [2]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pyspark.pandas as ps

import os

In [8]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics

In [3]:
# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 6
MEMORY_PER_EXECUTOR_MB = 4096 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "4g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 6, 4096]


26/06/02 15:23:00 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.79.162 instead (on interface wlo1)
26/06/02 15:23:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 15:23:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Сессия создана.
Driver Memory Config: 4g
Executor Memory Config: 4g


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 562955
Executor ID: Driver/Local, PID: 562961
Executor ID: Driver/Local, PID: 563006
Executor ID: Driver/Local, PID: 562999


In [4]:
n = 5000  # увеличьте для теста IVF-индексов
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [13]:
nn = 200
index_df = pd.DataFrame(
    {
        '0': np.random.randint(0, 5000, nn),
        '1': np.random.randint(0, 5000, nn),
        '2': np.random.randint(0, 5000, nn),
        '3': np.random.randint(0, 5000, nn),
        '4': np.random.randint(0, 5000, nn),
        'group': [0] * (nn//4) + [1] * (nn//4) + [2] * (nn//4) + [3] * (nn//4)
    }
)

# index_df = pd.concat([index_df, pd.DataFrame(data={
#     '0': np.nan,
#     '1': np.nan,
#     '2': np.nan,
#     '3': np.nan,
#     '4': np.nan,
#     'group': np.nan
# }, index=[0])]).reset_index(drop=True)
# index_df

In [ ]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3")
            # .config("spark.jars.packages", "ch.cern.sparkmeasure:spark-measure_2.12:0.23") 
            .getOrCreate()
          )

In [ ]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    # data=session.createDataFrame(index_df),
    data=sp_s.createDataFrame(index_df),
    # session=session
    session=sp_s
)

# index_ds

In [6]:


# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

# dataset_1 = Dataset(
#     roles=roles,
#     data=spark_df,
#     # data=session.createDataFrame(df),
#     # data=df,
#     backend=BackendsEnum.spark,
#     session=sp_s,
#     # session=session
# )

# dataset_2 = Dataset(
#     roles=roles,
#     data=spark_df,
#     # data=session.createDataFrame(df),
#     # data=df,
#     backend=BackendsEnum.spark,
#     session=sp_s,
#     # session=session
# )

# dataset

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [9]:
experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        # Bias(
        #     grouping_role=TreatmentRole(), 
        #     target_roles=[TargetRole()]
        # ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        OnRoleExperiment(
            executors=[
                TTest(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole(),
                )
            ],
            role=FeatureRole()
        )
    ]
)
result = experiment.execute(ExperimentData(dataset))

DummyEncoder


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


executor.key = DummyEncoder┴┴┆feat_cat_C┆; dt = 0.9348c
MahalanobisDistance


26/06/02 15:24:04 WARN AttachDistributedSequenceExec: clean up cached RDD(49) in AttachDistributedSequenceExec(154)
26/06/02 15:24:05 WARN AttachDistributedSequenceExec: clean up cached RDD(65) in AttachDistributedSequenceExec(359)
26/06/02 15:24:05 WARN AttachDistributedSequenceExec: clean up cached RDD(79) in AttachDistributedSequenceExec(716)
26/06/02 15:24:06 WARN AttachDistributedSequenceExec: clean up cached RDD(87) in AttachDistributedSequenceExec(740)
26/06/02 15:24:07 WARN AttachDistributedSequenceExec: clean up cached RDD(116) in AttachDistributedSequenceExec(1564)
26/06/02 15:24:07 WARN AttachDistributedSequenceExec: clean up cached RDD(124) in AttachDistributedSequenceExec(1588)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26

executor.key = MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||┆feat_cat_B┆', 'DummyEncoder||┆feat_cat_C┆']; dt = 13.0058c
TypeCaster
executor.key = TypeCaster┴┴; dt = 0.0070c
FaissNearestNeighbors


...

executor.key = TTest┴┴; dt = 22.0434c
StatsTTest


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is l

executor.key = TTest┴┴; dt = 21.0247c
StatsTTest


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/02 15:26:17 WARN AttachDistributedSequenceExec: clean up cached RDD(9155) in AttachDistributedSequenceExec(416620)
26/06/02 15:26:19 WARN AttachDistributedSequenceExec: cle

executor.key = TTest┴┴; dt = 3.5513c
executor.key = OnRoleExperiment┴┴; dt = 46.6626c


In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [14]:
result.additional_fields

26/06/02 15:28:43 WARN AttachDistributedSequenceExec: clean up cached RDD(10187) in AttachDistributedSequenceExec(459962)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/02 15:28:45 WARN AttachDistributedSequenceExec: clean up cached RDD(10326) in AttachDistributedSequenceExec(465898)
26/06/02 15:28:45 WARN AttachDistributedSequenceExec: clean up cached RDD(10332) in AttachDistributedSequenceExec(466174)
26/06/02 15:28:45 WARN AttachDistributedSequenceExec: clean up cached RDD(10338) in AttachDistributedSequenceExec(466223)
26/06/02 15:28:47 WARN AttachDistributedSequenceExec: clean up cached RDD(10478) in AttachDistributedSequenceExec(472110)
26/06/02 15:28:47 WARN AttachDistributedSequenceExec: clean up cached RDD(10484) in AttachDi

,DummyEncoder┴┴┆feat_cat_B┆,DummyEncoder┴┴┆feat_cat_C┆,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,MatchingMetrics┴┴
1255,0,0,1255,4680,96.200602
1250,0,0,1250,614,98.70928
1256,0,1,954,3112,90.70889
1252,0,1,1252,3276,90.151151
1253,0,0,1253,3936,103.374963
...,...,...,...,...,...
4995,0,1,311,0,104.077057
4996,0,1,340,3133,99.23931
4997,0,1,4541,2823,98.780726
4998,0,1,3582,255,103.574326


In [12]:
result.analysis_tables

{'StatsTTest┴┴feat_num_1┆stats':    mean┆feat_num_1  std┆feat_num_1  count┆feat_num_1  mean┆feat_num_1_matched  std┆feat_num_1_matched  count┆feat_num_1_matched
 1         9.973996        3.051757            2003.0                 9.977416                3.013007                    2003.0
 0        10.055198        2.982106            2997.0                10.055198                2.982106                    2997.0
 
 2 rows × 6 columns,
 'StatsTTest┴┴feat_num_1':                p-value  statistic  pass
 1┆feat_num_1  0.971535  -0.035686   0.0
 0┆feat_num_1  1.000000   0.000000   0.0
 
 2 rows × 3 columns,
 'StatsTTest┴┴feat_num_2┆stats':    mean┆feat_num_2  std┆feat_num_2  count┆feat_num_2  mean┆feat_num_2_matched  std┆feat_num_2_matched  count┆feat_num_2_matched
 1        -1.963166        1.488729            2003.0                -1.958072                1.470318                    2003.0
 0        -2.013851        1.492787            2997.0                -2.013851                1.

In [11]:
result.variables["MatchingMetrics┴┴['target', 'target_matched']"]

{'ATT': [0.26362007093594203,
  0.27360427695698514,
  0.33529235577219474,
  -0.27264431189974886,
  0.799884453771633],
 'ATC': [-0.05693055777925273,
  0.3316560259428989,
  0.8637084272314186,
  -0.7069763686273345,
  0.5931152530688291],
 'ATE': [0.07148202408405428,
  0.2610416289146767,
  0.7842123389164402,
  -0.440159568588712,
  0.5831236167568206]}

In [15]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [ ]:
sp_s.stop()